# Connect to PostgreSQL

In [66]:
import psycopg2

try:
   # Connect to PostgreSQL
   connection = psycopg2.connect(
       dbname="Evolution",
       user="ev",
       password="Temp@123",
       host="192.168.4.51", # or your server's IP address
       port="5432" # default PostgreSQL port
   )
   print("Connection established successfully!")
except Exception as e:
   print(f"Error: {e}")


Connection established successfully!


In [67]:
cur = connection.cursor()

# Load the data

In [68]:
import pandas as pd
import os
import json


In [69]:
table_columns_path = 'table_columns.json'
with open(table_columns_path, 'r', encoding='utf-8-sig') as file:
                table_columns = json.load(file)

In [70]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)

In [41]:
for x in table_columns["Bookings"]:
    print(x)
    

Id
UnitId
OriginalPrice
AgreedPrice
TransferredToDeal
ZohoReference
CustomerId
ProjectId
Stage
AgentId
ManagerId
HeadOfSalesId
CSOId
Layout
BrokerId
CreatedBy
CreationDate
IsDeleted
PaymentLinkId
BookingReferenceId
DraftedAmount


In [42]:
def pre_row(row, table_name):
    for col in table_columns[table_name]:
        if col not in row or row[col] == '-' or row[col] == '' or row[col] == ' ':
            
            if table_columns[table_name][col] == "int" or table_columns[table_name][col] == "float":
                    row[col] = ' 0 '
            else:
                 row[col] = ' Null '
    return row

In [71]:
def row2text():

	return eval(text_data)
        

In [72]:
entrys = [ "Bookings.json", "PaymentLinks.json"]

In [73]:
directory = "C:/Users/maryam.maksour/OneDrive - AL BAYARI/Desktop/RAG/DATA"


# Load the embedding model

In [74]:
from langchain_ollama import OllamaEmbeddings

emb = OllamaEmbeddings(model="bge-large", base_url="http://192.168.43.220:11435")

def generate_embedding(text): 
    text = str(text)
    embedding = emb.embed_query(text)
    return embedding

# Bookings

In [82]:
# Delete the table
x = cur.execute("""DROP TABLE Bookings""")

connection.commit()
connection.rollback()

In [81]:

connection.commit()
connection.rollback()

In [83]:

x = cur.execute("""CREATE TABLE Bookings (
    Id                  INTEGER PRIMARY KEY,
    UnitId              INTEGER,
    OriginalPrice       DOUBLE PRECISION,
    AgreedPrice         DOUBLE PRECISION,
    TransferredToDeal   BOOLEAN,
    CustomerId          INTEGER,
    ProjectId           INTEGER,
    Stage               TEXT,
    AgentId             INTEGER,
    ManagerId           INTEGER,
    HeadOfSalesId       INTEGER,
    CSOId               INTEGER,
    Layout              TEXT,
    BrokerId            INTEGER,
    CreatedBy           TEXT,
    CreationDate        TEXT,
    IsDeleted           BOOLEAN,
    PaymentLinkId       INTEGER,
    BookingReferenceId  TEXT,
    DraftedAmount       DOUBLE PRECISION,
    row_text TEXT,
    embedding VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

In [84]:
 
directory = "C:/Users/maryam.maksour/OneDrive - AL BAYARI/Desktop/RAG/DATA"

full_path = os.path.join(directory, "Bookings.json")
file_name = "Bookings"
text_data = text_data[file_name]

TypeError: string indices must be integers, not 'str'

In [85]:
text_data

'f" Bookings with id {row[\'Id\']} for Unit id {row[\'UnitId\']}. Original price {row[\'OriginalPrice\']}, agreed price {row[\'AgreedPrice\']}. transferred to deal: {row[\'TransferredToDeal\']}. customer id {row[\'CustomerId\']}. project id {row[\'ProjectId\']}. current stage {row[\'Stage\']}. agent id {row[\'AgentId\']}, manager id {row[\'ManagerId\']}, head of sales id {row[\'HeadOfSalesId\']}, CSO id {row[\'CSOId\']}. layout: {row[\'Layout\']}. broker id {row[\'BrokerId\']}. created by {row[\'CreatedBy\']} on {row[\'CreationDate\']}. is deleted: {row[\'IsDeleted\']}. payment link id {row[\'PaymentLinkId\']}. booking reference id {row[\'BookingReferenceId\']}. drafted amount {row[\'DraftedAmount\']}. "'

In [86]:

if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
                data = json.load(file)
                
                i = 0
                try:
                   for row in data[file_name]:
                         row2 = pre_row(row, file_name)
                         txt = row2text( )
                         print(txt)
                         embedding = generate_embedding(txt)
                          
                         cur.execute("""INSERT INTO Bookings (  Id, UnitId, OriginalPrice, AgreedPrice, TransferredToDeal, CustomerId,
										ProjectId, Stage,  AgentId,  ManagerId, HeadOfSalesId, CSOId, Layout,
										BrokerId,  CreatedBy, CreationDate, IsDeleted, PaymentLinkId, BookingReferenceId, DraftedAmount,row_text, embedding
									) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)""",  
                         ( row2['Id'], int(row2['UnitId']), float(row2['OriginalPrice']) , float(row2['AgreedPrice']),row2['TransferredToDeal'],
                           int(row2['CustomerId']), int(row2['ProjectId']), row2['Stage'], int(row2['AgentId']),int(row2['ManagerId']),  
                                       int(row2['HeadOfSalesId']),int(row2['CSOId']),
									   row2['Layout'], int(row2['BrokerId']), row2['CreatedBy'],row2['CreationDate'],row2['IsDeleted'],
                                             row2['PaymentLinkId'],row2['BookingReferenceId'], int(row2['DraftedAmount']),
                            txt,embedding ))
                         if (i+1) % 100 == 0:
                                 connection.commit()
                         i += 1
						 
                except Exception as e:
                     print(file_name)
                     print(e)
       


 Bookings with id 188 for Unit id 4235. Original price 4800.0, agreed price 4800.0. transferred to deal: False. customer id 2422. project id 32. current stage Draft. agent id 542, manager id 523, head of sales id 517, CSO id 516. layout: 1 Bedroom. broker id 6. created by  Null  on 2025-09-12T16:53:45.0279936. is deleted: False. payment link id 81. booking reference id 5AC4FEBC-A8C3-42B0-84F3-8D81CCB7CBEF. drafted amount 0.0. 
 Bookings with id 189 for Unit id 4325. Original price 6000.0, agreed price 6000.0. transferred to deal: False. customer id 2423. project id 32. current stage Draft. agent id 542, manager id 523, head of sales id 517, CSO id 516. layout: 2 Bedroom. broker id 6. created by  Null  on 2025-09-12T17:07:17.0454093. is deleted: False. payment link id 82. booking reference id 9EDA330F-C379-4421-845B-7B5873F09BF9. drafted amount 0.0. 
 Bookings with id 190 for Unit id 4225. Original price 84706.62, agreed price 84706.62. transferred to deal: False. customer id 2424. proj

# PaymentLinks

In [88]:
# Delete the table
x = cur.execute("""DROP TABLE payment_links""")

connection.commit()
connection.rollback()

UndefinedTable: table "payment_links" does not exist


In [89]:

connection.commit()
connection.rollback()

In [90]:

x = cur.execute("""
CREATE TABLE IF NOT EXISTS PaymentLinks (
    id                          INTEGER PRIMARY KEY,
    amount                      NUMERIC(18,2),
    currency                    TEXT      ,
    payment_type                TEXT      ,
    payment_url                 TEXT      ,
    status                      TEXT      ,
    expiry_date                 TIMESTAMP ,
    ref_no                      TEXT      ,
    payment_link_reference_id   UUID      ,
    project_id                  INTEGER   ,
    agent_id                    INTEGER   ,
    created_by                  TEXT      ,
    creation_date               TIMESTAMP  ,
    is_deleted                  BOOLEAN      ,
    is_splited                  BOOLEAN      ,
    remaining_amount            NUMERIC(18,2) ,
    service_charge_percentage   NUMERIC(5,2)    ,
    row_text                    TEXT            ,
    embedding                   VECTOR(1024)     

);
""")

connection.commit()
connection.rollback()

In [91]:
directory = "C:/Users/maryam.maksour/OneDrive - AL BAYARI/Desktop/RAG/DATA"

text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
full_path = os.path.join(directory, "PaymentLinks.json")
file_name = "PaymentLinks"
text_data = text_data[file_name]

In [92]:

if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
                data = json.load(file)
                
                i = 0
                try:
                   for row in data[file_name]:
                         row2 = pre_row(row, file_name)
                         txt = row2text( )
                         embedding = generate_embedding(txt)
                          
                         cur.execute("""INSERT INTO PaymentLinks (  id, amount, currency, payment_type, payment_url, status,
										expiry_date, ref_no,  payment_link_reference_id,  project_id, agent_id, created_by, creation_date,
										is_deleted,  is_splited, remaining_amount, service_charge_percentage,row_text, embedding
									) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)""",  
                         ( row2['Id'], float(row2['Amount']), row2['Currency'] , row2['PaymentType'],
                            row2['PaymentUrl'] , row2['Status'] , row2['ExpiryDate'],  row2['RefNo'] , row2['PaymentLinkReferenceId'] ,  
                                       int(row2['ProjectId']),int(row2['AgentId']),
									    row2['CreatedBy'],row2['CreationDate'],row2['IsDeleted'],
                                                          row2['IsSplited'], float(row2['RemainingAmount']),
                                              float(row2['ServiceChargePercentage']),
                            txt,embedding ))
                         if (i+1) % 100 == 0:
                                 connection.commit()
                         i += 1
						 
                except Exception as e:
                     print(file_name)
                     print(e)
       
